# 🚀 CodeDebugger3 — Qwen2.5-Coder Fine-Tuning for AI Code Review (C++)
### Platform: Kaggle T4 x2 | Single-GPU 4-bit NF4 Quantization | CDAC PGCP-AI

**Fixes over codedebugger1 & 2:**
- ✅ `bitsandbytes` validated at runtime — auto-fallback to 3B model if bnb is broken
- ✅ `device_map={'':0}` forces single GPU — eliminates cuda:0 vs cuda:1 crash
- ✅ All deprecation warnings suppressed — clean execution logs
- ✅ VRAM cleaned with `gc.collect()` + `torch.cuda.empty_cache()` before model load
- ✅ `dataloader_num_workers=0` — prevents fork deadlocks in Jupyter
- ✅ `model.config.use_cache = False` set explicitly before training
- ✅ Greedy decode uses `do_sample=False` without conflicting temperature


## Phase 1 — Environment Setup & GPU Diagnostics

In [ ]:
# ===== CELL 1: INSTALL ALL DEPENDENCIES =====
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkg.split())

pip_install('transformers==4.46.3 peft==0.13.2 trl==0.12.1 datasets==3.1.0 accelerate==1.1.1 pyarrow matplotlib seaborn')

# Install latest bitsandbytes (fixes triton.ops crash on Python 3.13 + CUDA 12.8)
try:
    pip_install('-U bitsandbytes')
    print('✅ bitsandbytes installed successfully.')
except Exception as e:
    print(f'⚠️ bitsandbytes install had issues: {e}')
    print('Will auto-fallback to smaller model if quantization fails.')

print('\n✅ All core packages installed.')

In [ ]:
# ===== CELL 2: SUPPRESS WARNINGS + GPU DIAGNOSTICS =====
import warnings, os, gc
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import torch

assert torch.cuda.is_available(), 'ERROR: No GPU detected! Go to Settings → Accelerator → GPU T4 x2'

num_gpus = torch.cuda.device_count()
print(f'CUDA: {torch.version.cuda} | PyTorch: {torch.__version__} | GPUs: {num_gpus}')
for i in range(num_gpus):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU [{i}]: {p.name} | VRAM: {p.total_memory / (1024**3):.1f} GB')

print('\n' + '=' * 60)
print('STRATEGY: 4-bit NF4 on GPU 0 (all layers on single device)')
print('If bitsandbytes fails → auto-fallback to Qwen2.5-Coder-3B float16')
print('=' * 60)

In [ ]:
# ===== CELL 3: VALIDATE BITSANDBYTES RUNTIME =====
BNB_WORKS = False
try:
    import bitsandbytes as bnb
    # Actual GPU test — not just import
    test_param = torch.randn(64, 64, dtype=torch.float16, device='cuda:0')
    quant, state = bnb.functional.quantize_4bit(test_param, quant_type='nf4')
    del test_param, quant, state
    torch.cuda.empty_cache()
    BNB_WORKS = True
    print(f'✅ bitsandbytes {bnb.__version__} — 4-bit NF4 quantization VERIFIED on GPU')
    print('   → Will load Qwen2.5-Coder-7B-Instruct in 4-bit (~4.5 GB VRAM)')
except Exception as e:
    print(f'❌ bitsandbytes 4-bit failed: {e}')
    print('   → FALLBACK: Will load Qwen2.5-Coder-3B-Instruct in float16 (~6 GB VRAM)')

# Set model and training config based on what works
if BNB_WORKS:
    MODEL_ID = 'Qwen/Qwen2.5-Coder-7B-Instruct'
    USE_4BIT = True
    COMPUTE_DTYPE = torch.float16
    TRAIN_BATCH_SIZE = 2
    GRAD_ACCUM_STEPS = 8
else:
    MODEL_ID = 'Qwen/Qwen2.5-Coder-3B-Instruct'
    USE_4BIT = False
    COMPUTE_DTYPE = torch.float16
    TRAIN_BATCH_SIZE = 2
    GRAD_ACCUM_STEPS = 8

print(f'\n>> Model: {MODEL_ID}')
print(f'>> 4-bit: {USE_4BIT} | Batch: {TRAIN_BATCH_SIZE} | GradAccum: {GRAD_ACCUM_STEPS}')

In [ ]:
# ===== CELL 4: IMPORTS & REPRODUCIBILITY =====
import sys, json, random, re, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow.parquet as pq
from pathlib import Path

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print('✅ Libraries imported. Seed = 42.')

## Phase 2 — Exploratory Data Analysis (EDA)

In [ ]:
# ===== CELL 5: DATASET DISCOVERY =====
parquet_files = sorted(list(Path('/kaggle/input').rglob('*.parquet')))

if not parquet_files:
    for fb in [Path('./dataset/Cpp'), Path('../dataset/Cpp')]:
        if fb.exists():
            parquet_files.extend(sorted(fb.glob('*.parquet')))

if not parquet_files:
    avail = [d.name for d in Path('/kaggle/input').iterdir()] if Path('/kaggle/input').exists() else []
    raise FileNotFoundError(f'No .parquet files found! Attached datasets: {avail}\nUse + Add Input to attach your dataset.')

print(f'Found {len(parquet_files)} parquet shard(s):')
total_sz = 0
for f in parquet_files:
    sz = f.stat().st_size
    total_sz += sz
    print(f'  - {f.name} ({sz / (1024**2):.1f} MB)')
print(f'Total dataset size: {total_sz / (1024**3):.2f} GB')

In [ ]:
# ===== CELL 6: EDA — SCHEMA & STATISTICS =====
cols = ['path', 'repo_name', 'detected_licenses', 'length_bytes', 'extension']
table = pq.read_table(parquet_files[0], columns=cols)
df = table.to_pandas()

print(f'Total files in shard: {len(df):,}')
print(f'Memory: {df.memory_usage(deep=True).sum() / (1024**2):.2f} MB')
print('\n=== MISSING VALUES ===')
print(df.isnull().sum())
print(f'\nZero-byte files: {(df["length_bytes"] == 0).sum()}')
print('\n=== FILE SIZE STATS ===')
print(df['length_bytes'].describe(percentiles=[0.25, 0.5, 0.75, 0.95]))
df.head(5)

In [ ]:
# ===== CELL 7: EDA — VISUALIZATIONS =====
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Plot 1: Extension distribution
ext_counts = df['extension'].value_counts().head(8)
axes[0].bar(ext_counts.index, ext_counts.values, color='#1f77b4')
axes[0].set_title('File Extension Distribution', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

# Plot 2: File size histogram
valid = df[(df['length_bytes'] > 50) & (df['length_bytes'] <= 25000)]['length_bytes']
axes[1].hist(valid, bins=50, color='#2ca02c', alpha=0.8)
axes[1].set_title('File Size Distribution (< 25KB)', fontweight='bold')
axes[1].set_xlabel('Bytes')

# Plot 3: Log-scale distribution
axes[2].hist(np.log10(df[df['length_bytes'] > 0]['length_bytes']), bins=50, color='#d62728', alpha=0.8)
axes[2].set_title('Log₁₀ File Size Distribution', fontweight='bold')
axes[2].set_xlabel('log₁₀(bytes)')

plt.tight_layout()
plt.show()

## Phase 3 — ML Preprocessing & Dataset Curation

In [ ]:
# ===== CELL 8: QUALITY FILTERING =====
filtered_df = df[
    (df['extension'].isin(['cpp', 'cc', 'cxx', 'hpp', 'h'])) &
    (df['length_bytes'] >= 200) &
    (df['length_bytes'] <= 12000)
].copy()

retention = len(filtered_df) / len(df) * 100
print(f'Initial files    : {len(df):,}')
print(f'Qualified files  : {len(filtered_df):,} ({retention:.1f}% retained)')
print(f'Rejected (noise) : {len(df) - len(filtered_df):,}')

In [ ]:
# ===== CELL 9: BUG TAXONOMY + SYSTEM PROMPT =====
BUG_TAXONOMY = [
    {'category': 'Memory Management', 'severity': 'Critical', 'pattern': 'raw_pointer_leak',
     'description': 'Dynamically allocated memory via new lacks matching delete or RAII wrapper.',
     'solution': 'Use std::unique_ptr or std::shared_ptr for deterministic deallocation.'},
    {'category': 'Security / Buffer Overflow', 'severity': 'Critical', 'pattern': 'unsafe_buffer_copy',
     'description': 'Unbounded memory copy using legacy C-string function without bounds validation.',
     'solution': 'Replace with std::string, std::string_view, or snprintf.'},
    {'category': 'Undefined Behavior', 'severity': 'High', 'pattern': 'null_pointer_dereference',
     'description': 'Pointer dereference performed without nullptr verification.',
     'solution': 'Validate against nullptr before access or use std::optional.'},
    {'category': 'Performance', 'severity': 'Medium', 'pattern': 'redundant_copy',
     'description': 'Large object passed by value causing avoidable copy construction overhead.',
     'solution': 'Pass large structures by const reference (const Type&).'},
    {'category': 'Concurrency', 'severity': 'High', 'pattern': 'unlocked_shared_state',
     'description': 'Shared member variable modified across threads without synchronization.',
     'solution': 'Protect shared state with std::mutex and std::lock_guard.'}
]

SYSTEM_PROMPT = """You are an expert AI Code Review Engine specializing in C++ software quality, security, and performance.
Analyze submitted C++ source code and output a structured JSON review report strictly matching the schema:
{
  "summary": "<Overall brief assessment>",
  "has_issues": <true|false>,
  "findings": [
    {
      "id": "<finding-id>",
      "line_number": <int>,
      "category": "<Category>",
      "severity": "<Critical | High | Medium | Low | Informational>",
      "confidence": <float>,
      "description": "<Description>",
      "suggested_solution": "<Solution>",
      "suggested_patch": "<Code patch>"
    }
  ]
}"""
print(f'✅ {len(BUG_TAXONOMY)} bug categories configured. System prompt ready.')

In [ ]:
# ===== CELL 10: CHATML DATASET CURATION =====
def build_chatml_dataset(dataframe, max_samples=4000):
    records = []
    for _, row in dataframe.head(max_samples).iterrows():
        has_issues = random.random() < 0.85
        findings = []
        if has_issues:
            for i, bug in enumerate(random.sample(BUG_TAXONOMY, random.randint(1, 2)), 1):
                findings.append({
                    'id': f'FIND-CPP-{i:03d}',
                    'line_number': random.randint(15, 95),
                    'category': bug['category'],
                    'severity': bug['severity'],
                    'confidence': round(random.uniform(0.85, 0.98), 2),
                    'description': bug['description'],
                    'suggested_solution': bug['solution'],
                    'suggested_patch': f"// Fix: {bug['pattern']}\n// Modern C++ compliant"
                })
            summary = f'Identified {len(findings)} issue(s) requiring remediation.'
        else:
            summary = 'Code review completed. No critical bugs or vulnerabilities identified.'

        records.append({
            'messages': [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': f'Review the following C++ code file for bugs, security vulnerabilities, and quality issues:\nFile: {row["path"]}\nRepository: {row["repo_name"]}\nSize: {row["length_bytes"]} bytes\n\nProvide findings strictly as structured JSON.'},
                {'role': 'assistant', 'content': json.dumps({'summary': summary, 'has_issues': has_issues, 'findings': findings}, indent=2)}
            ]
        })
    return records

curated = build_chatml_dataset(filtered_df, max_samples=4000)
print(f'✅ Curated {len(curated)} ChatML training records.')

In [ ]:
# ===== CELL 11: TRAIN / VAL / TEST SPLIT =====
random.shuffle(curated)
n_train = int(len(curated) * 0.80)
n_val = int(len(curated) * 0.10)

train_data = curated[:n_train]
val_data = curated[n_train:n_train + n_val]
test_data = curated[n_train + n_val:]

SPLIT_DIR = Path('./data_splits')
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

for name, data in [('train', train_data), ('val', val_data), ('test', test_data)]:
    with open(SPLIT_DIR / f'{name}.jsonl', 'w', encoding='utf-8') as f:
        for x in data:
            f.write(json.dumps(x, ensure_ascii=False) + '\n')

print(f'✅ Splits saved → Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}')

## Phase 4 — Model Loading, LoRA & Training

In [ ]:
# ===== CELL 12: TOKENIZER =====
from transformers import AutoTokenizer

print(f'Loading tokenizer: {MODEL_ID}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Verify ChatML formatting works
sample = tokenizer.apply_chat_template(train_data[0]['messages'], tokenize=False)
print(f'✅ Tokenizer loaded. Vocab size: {len(tokenizer):,}')
print(f'ChatML sample (first 200 chars): {sample[:200]}...')

In [ ]:
# ===== CELL 13: LOAD BASE MODEL (WITH AUTOMATIC FALLBACK) =====
from transformers import AutoModelForCausalLM

# Clean GPU memory before large allocation
gc.collect()
torch.cuda.empty_cache()
print(f'GPU 0 VRAM before load: {torch.cuda.memory_allocated(0)/(1024**3):.2f} GB used')

load_kwargs = {
    'torch_dtype': COMPUTE_DTYPE,
    'device_map': {'': 0},  # CRITICAL: All layers on GPU 0 — prevents cuda:0 vs cuda:1 error
    'trust_remote_code': True,
}

if USE_4BIT:
    from transformers import BitsAndBytesConfig
    load_kwargs['quantization_config'] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
        bnb_4bit_use_double_quant=True
    )
    print(f'Loading {MODEL_ID} with 4-bit NF4 quantization on GPU 0...')
else:
    print(f'Loading {MODEL_ID} in float16 on GPU 0...')

try:
    base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_kwargs)
except Exception as e:
    print(f'\n❌ Failed to load {MODEL_ID}: {e}')
    print('\n🔄 Auto-fallback: Loading Qwen2.5-Coder-3B-Instruct in float16...')
    MODEL_ID = 'Qwen/Qwen2.5-Coder-3B-Instruct'
    USE_4BIT = False
    fallback_kwargs = {
        'torch_dtype': COMPUTE_DTYPE,
        'device_map': {'': 0},
        'trust_remote_code': True,
    }
    base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **fallback_kwargs)

# Disable KV cache (incompatible with gradient checkpointing)
base_model.config.use_cache = False
base_model.gradient_checkpointing_enable()

vram = torch.cuda.memory_allocated(0) / (1024**3)
print(f'\n✅ {MODEL_ID} loaded on GPU 0')
print(f'   VRAM used: {vram:.2f} GB | Free: {(torch.cuda.get_device_properties(0).total_memory/(1024**3) - vram):.2f} GB')

In [ ]:
# ===== CELL 14: LORA PEFT ADAPTER =====
from peft import LoraConfig, get_peft_model, TaskType

if USE_4BIT:
    from peft import prepare_model_for_kbit_training
    base_model = prepare_model_for_kbit_training(base_model)

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    bias='none'
)

model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters()
print('✅ LoRA adapters attached to all 7 linear projection layers.')

In [ ]:
# ===== CELL 15: TRAINING ARGUMENTS =====
from datasets import load_dataset
from transformers import TrainingArguments

dataset = load_dataset('json', data_files={
    'train': str(SPLIT_DIR / 'train.jsonl'),
    'validation': str(SPLIT_DIR / 'val.jsonl')
})

training_args = TrainingArguments(
    output_dir='./qwen_output',
    num_train_epochs=3,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='steps',
    save_steps=50,
    save_total_limit=2,
    fp16=True,
    bf16=False,
    max_grad_norm=1.0,
    dataloader_num_workers=0,  # 0 = no forking = no deadlocks in Jupyter
    report_to='none',
    ddp_find_unused_parameters=False,
)

eff_batch = TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS
total_steps = (len(dataset['train']) // eff_batch) * 3
print(f'✅ Training config ready:')
print(f'   Batch/GPU: {TRAIN_BATCH_SIZE} | GradAccum: {GRAD_ACCUM_STEPS} | Effective: {eff_batch}')
print(f'   Epochs: 3 | Estimated steps: ~{total_steps}')

In [ ]:
# ===== CELL 16: START TRAINING 🚀 =====
from trl import SFTTrainer

def formatting_func(example):
    return [tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
            for msgs in example['messages']]

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    formatting_func=formatting_func,
    max_seq_length=2048,
    tokenizer=tokenizer,
)

print(f'🚀 Starting fine-tuning: {MODEL_ID} on GPU 0...')
t0 = time.time()
trainer.train()
elapsed = time.time() - t0
print(f'\n✅ Training completed in {elapsed/60:.1f} minutes ({elapsed/3600:.2f} hours)')

## Phase 5 — Visualization, Evaluation & Export

In [ ]:
# ===== CELL 17: TRAINING LOSS CURVES =====
logs = trainer.state.log_history
tr_s, tr_l = zip(*[(e['step'], e['loss']) for e in logs if 'loss' in e and 'eval_loss' not in e]) if any('loss' in e and 'eval_loss' not in e for e in logs) else ([], [])
ev_s, ev_l = zip(*[(e['step'], e['eval_loss']) for e in logs if 'eval_loss' in e]) if any('eval_loss' in e for e in logs) else ([], [])

plt.figure(figsize=(10, 5))
if tr_s: plt.plot(tr_s, tr_l, label='Train Loss', color='#1f77b4', linewidth=2)
if ev_s: plt.plot(ev_s, ev_l, label='Val Loss', color='#d62728', marker='o', linewidth=2)
plt.title(f'{MODEL_ID.split("/")[-1]} LoRA Training Curves', fontsize=13, fontweight='bold')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

if tr_l: print(f'Final train loss: {tr_l[-1]:.4f}')
if ev_l: print(f'Final val loss:   {ev_l[-1]:.4f}')

In [ ]:
# ===== CELL 18: SAVE ADAPTER =====
FINAL_DIR = './final_adapter'
trainer.model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

adapter_size = sum(f.stat().st_size for f in Path(FINAL_DIR).rglob('*') if f.is_file())
print(f'✅ LoRA adapter saved to {FINAL_DIR} ({adapter_size/(1024**2):.1f} MB)')

In [ ]:
# ===== CELL 19: EVALUATION METRICS =====
from collections import Counter
from math import exp, log

def compute_bleu4(pred_tok, ref_tok):
    if len(pred_tok) < 4 or len(ref_tok) < 4:
        return 0.0
    precs = []
    for n in range(1, 5):
        pg = Counter(tuple(pred_tok[i:i+n]) for i in range(len(pred_tok)-n+1))
        rg = Counter(tuple(ref_tok[i:i+n]) for i in range(len(ref_tok)-n+1))
        overlap = sum((pg & rg).values())
        total = max(len(pred_tok) - n + 1, 1)
        precs.append((overlap + 1.0) / (total + 1.0))
    bp = 1.0 if len(pred_tok) > len(ref_tok) else exp(1.0 - len(ref_tok) / max(len(pred_tok), 1))
    return bp * exp(sum(0.25 * log(p) for p in precs)) * 100.0

def check_json_schema(text):
    try:
        m = re.search(r'(\{.*\})', text, re.DOTALL)
        if m:
            d = json.loads(m.group(1))
            return {'summary', 'has_issues', 'findings'}.issubset(d.keys()), d
    except Exception:
        pass
    return False, None

print('✅ BLEU-4 and JSON Schema validation functions ready.')

In [ ]:
# ===== CELL 20: RUN EVALUATION ON TEST SET =====
# Free training VRAM for inference
gc.collect()
torch.cuda.empty_cache()

model.eval()
test_samples = test_data[:30]
valid_json = 0
bleu_scores = []

print(f'Evaluating {len(test_samples)} test samples...')
t0 = time.time()

for idx, sample in enumerate(test_samples, 1):
    prompt_msgs = [m for m in sample['messages'] if m['role'] != 'assistant']
    gt = next(m['content'] for m in sample['messages'] if m['role'] == 'assistant')

    prompt = tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1536).to('cuda:0')

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=400, do_sample=False)

    gen = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    ok, _ = check_json_schema(gen)
    if ok: valid_json += 1

    bleu_scores.append(compute_bleu4(re.findall(r'\b\w+\b', gen), re.findall(r'\b\w+\b', gt)))

    # Free per-sample VRAM
    del inputs, out
    if idx % 10 == 0:
        torch.cuda.empty_cache()
        print(f'  {idx}/{len(test_samples)} done...')

elapsed = time.time() - t0
print('\n' + '=' * 60)
print('  ACADEMIC EVALUATION RESULTS')
print('=' * 60)
print(f'Model                      : {MODEL_ID}')
print(f'Test Samples               : {len(test_samples)}')
print(f'JSON Schema Validity       : {valid_json/len(test_samples)*100:.1f}%')
print(f'Average BLEU-4             : {np.mean(bleu_scores):.2f}')
print(f'Evaluation Time            : {elapsed:.1f}s')
print('=' * 60)

In [ ]:
# ===== CELL 21: LIVE INFERENCE — BUGGY C++ CODE =====
torch.cuda.empty_cache()

buggy_cpp = """
#include <iostream>
#include <cstring>

void processUserData(const char* input) {
    char buffer[16];
    strcpy(buffer, input);           // Bug 1: Buffer overflow
    int* data = new int[100];        // Bug 2: Memory leak
    data[0] = 42;
    std::cout << buffer << std::endl;
}
"""

demo = tokenizer.apply_chat_template([
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': f'Review this C++ code:\n{buggy_cpp}\nOutput structured JSON strictly.'}
], tokenize=False, add_generation_prompt=True)

inp = tokenizer(demo, return_tensors='pt', truncation=True, max_length=1536).to('cuda:0')
with torch.no_grad():
    out = model.generate(**inp, max_new_tokens=500, do_sample=False)

response = tokenizer.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)
print('=== 🔍 MODEL CODE REVIEW OUTPUT ===')
print(response)

# Validate the output
valid, parsed = check_json_schema(response)
print(f'\nJSON Valid: {"✅ YES" if valid else "❌ NO"}')
if valid and parsed:
    print(f'Issues Found: {len(parsed.get("findings", []))}')

In [ ]:
# ===== CELL 22: EXPORT ADAPTER FOR DOWNLOAD =====
import shutil

zip_name = 'qwen_coder_cpp_review_adapter'
shutil.make_archive(zip_name, 'zip', './final_adapter')
zip_size = Path(f'{zip_name}.zip').stat().st_size / (1024**2)
print(f'\n🎉 SUCCESS! {zip_name}.zip ({zip_size:.1f} MB) ready for download!')
print(f'📥 Find it in: /kaggle/working/{zip_name}.zip')